# 3.5 Counting Techniques

**Goal:** count elementary outcomes correctly before converting counts to probabilities.

**Prerequisites:** factorials and set notation from Module 0, plus earlier Module 3 notebooks. **Runtime:** Python 3.10+, standard library only. Idealized examples and saved outputs are included; run cells in order.

## Contents
1. Fundamental counting principle
2. Factorials
3. Permutations
4. Combinations
5. Choosing the right formula
6. Counting probabilities and complements
7. AI/ML applications and limits of enumeration
8. Practice and solutions

## 1. Fundamental Counting Principle

If a process has $n_1$ choices at stage 1, $n_2$ choices at stage 2 for each first choice, and so on, then the number of complete paths is
$$n_1n_2\cdots n_k.$$
This is a statement about counts, not a probability-independence assumption. It requires that the stated number of available choices applies along every relevant path. If branches have different numbers of completions, sum their counts instead of using a single inappropriate product.

**Worked example:** 3 model families, 2 preprocessing options, and 4 thresholds produce $3\times2\times4=24$ configurations, provided every combination is permitted. If one family permits only one preprocessing option, count that branch separately.

The addition principle sums counts of **disjoint alternatives**. If alternatives overlap, use inclusion-exclusion to avoid double counting. Counting complete paths does not imply each path has equal probability of being chosen.

In [1]:
import math
from itertools import product, permutations, combinations, combinations_with_replacement
from fractions import Fraction
families = ["linear", "tree", "bayes"]
preprocessing = ["raw", "scaled"]
thresholds = [0.2, 0.4, 0.6, 0.8]
configurations = list(product(families, preprocessing, thresholds))
print("All compatible configurations:", len(configurations))
# Suppose the tree family allows only raw features in this toy search contract.
allowed = [c for c in configurations if c[0] != "tree" or c[1] == "raw"]
print("Restricted configurations:", len(allowed))
assert len(configurations) == 3*2*4
assert len(allowed) == 2*2*4+1*1*4

All compatible configurations: 24
Restricted configurations: 20


## 2. Factorials

For a nonnegative integer $n$,
$$n!=n(n-1)\cdots2\times1,\qquad0!=1.$$
The zero-factorial convention corresponds to one way to arrange no objects and to the empty product. Ordinary factorials here do not accept negative integers or noninteger inputs.

**Worked example:** 4 distinct books can be arranged in $4!=24$ orders. Factorials grow rapidly: 10! is 3,628,800 and 20! exceeds $2\times10^{18}$. Python integers store exact large values, but listing every arrangement quickly becomes infeasible.

Prefer `math.perm` and `math.comb` to dividing enormous factorials with floating-point `/`. Exact counting can be feasible even when enumeration is not; calculating a count is very different from generating every possibility.

In [2]:
for n in [0, 4, 10, 20]:
    print(f"{n}! = {math.factorial(n)}")
assert math.factorial(0) == 1
assert len(list(permutations("ABCD"))) == math.factorial(4)

0! = 1
4! = 24
10! = 3628800
20! = 2432902008176640000


## 3. Permutations

A permutation selects or arranges objects when **order matters**.

### Distinct objects, without replacement

For $0\le r\le n$,
$$P(n,r)=n(n-1)\cdots(n-r+1)=\frac{n!}{(n-r)!}.$$
There are $n$ choices for the first position, $n-1$ for the next, and so on. The empty ordered selection has count $P(n,0)=1$.

**Worked example:** assigning president and secretary among 5 people gives $P(5,2)=5\times4=20$. Choosing Alice then Bob differs from choosing Bob then Alice because the roles differ.

### Ordered positions, repetition allowed

If each of $r$ positions can use any of $n$ types independently of earlier choices as a counting constraint, the count is $n^r$.

**Worked example:** a 4-digit PIN permits digits 0–9 and leading zero, so there are $10^4=10000$ PINs. A four-digit positive integer does not permit leading zero, giving $9\times10^3=9000$. If a PIN also forbids repeated digits, its count is $P(10,4)=5040$.

### Arrangements with indistinguishable repeated objects

If all $n$ objects are arranged and identical groups have counts $n_1,\ldots,n_k$ summing to $n$, the number of distinct arrangements is
$$\frac{n!}{n_1!\cdots n_k!}.$$

**Worked example:** “LEVEL” has two Ls, two Es, and one V, giving $5!/(2!2!1!)=30$. Dividing removes swaps of identical letters that do not create distinct displayed strings. This is a different situation from allowing repetition independently in each position.

In [3]:
print("Two distinct officer roles:", math.perm(5, 2))
print("PINs with repetition:", 10**4, "without repetition:", math.perm(10, 4))
letters = "LEVEL"
arrangement_count = math.factorial(5)//(math.factorial(2)*math.factorial(2))
unique_words = set(permutations(letters))  # Safe only for this tiny example.
print("Distinct LEVEL arrangements:", arrangement_count)
assert len(unique_words) == arrangement_count == 30

Two distinct officer roles: 20
PINs with repetition: 10000 without repetition: 5040
Distinct LEVEL arrangements: 30


## 4. Combinations

A combination selects objects when **order does not matter**.

### Distinct objects, without replacement

$$\binom nr=\frac{n!}{r!(n-r)!}=\frac{P(n,r)}{r!},\qquad0\le r\le n.$$
Each unordered subset corresponds to $r!$ ordered selections, so divide them out. Symmetry gives $\binom nr=\binom n{n-r}$, and $\binom n0=\binom nn=1$.

**Worked example:** a 2-person committee from 5 people has $\binom52=10$ possibilities. Unlike officer roles, the committee {Alice,Bob} is unchanged if names are listed in reverse order.

### Unordered selections with repetition allowed

For $n\ge1$ types and $r\ge0$ selected objects,
$$\binom{n+r-1}{r}$$
counts multisets (equivalently, nonnegative count allocations across types). This follows by arranging $r$ item marks and $n-1$ separators, the “stars and bars” argument.

**Worked example:** choose 3 scoops from 4 flavors with repeated flavors allowed and order ignored: $\binom63=20$. This count does not imply the 20 multiset outcomes are equally likely under a process that draws each scoop independently. That distinction matters when using counts for probability.

In [4]:
people = ["A", "B", "C", "D", "E"]
print("Committee count:", math.comb(5, 2))
print("Officer assignment count:", math.perm(5, 2))
print("Three scoops, four flavors:", math.comb(4+3-1, 3))
assert len(list(combinations(people, 2))) == math.comb(5, 2)
assert len(list(combinations_with_replacement("ABCD", 3))) == math.comb(6, 3)
assert math.perm(5, 2) == math.comb(5, 2)*math.factorial(2)

Committee count: 10
Officer assignment count: 20
Three scoops, four flavors: 20


## 5. Choosing the right formula

| Order matters? | Replacement/repetition allowed? | Standard count | Assumptions |
|---|---|---|---|
| Yes | No | $P(n,r)$ | Distinct source objects, $r\le n$ |
| Yes | Yes | $n^r$ | Each position permits all $n$ types |
| No | No | $\binom nr$ | Distinct objects, subset size $r\le n$ |
| No | Yes | $\binom{n+r-1}{r}$ | Types can repeat; multiplicity matters |

For repeated indistinguishable objects arranged all at once, use the multinomial arrangement formula instead. Constraints such as “at least one,” “no adjacent repeats,” or “must include a particular item” often require cases, complements, or more specialized reasoning.

A reliable sequence is: define what counts as a distinct outcome, check order, check replacement, identify restrictions, count, then justify equal likelihood if a probability is requested.

## 6. From counts to probabilities

For a finite **uniform** sample space, $P(A)=|A|/|\Omega|$. Use consistent resolutions for numerator and denominator: unordered favorable hands over unordered total hands, or ordered favorable draws over ordered total draws. Mixing the two gives wrong answers.

**Worked example:** draw 5 cards uniformly without replacement from a standard 52-card deck with 4 aces. The chance of exactly two aces is
$$\frac{\binom42\binom{48}{3}}{\binom{52}{5}}\approx0.03993.$$
Choose 2 aces and 3 nonaces; do not allow an additional ace in the second factor. Unordered 5-card hands are equally likely under the uniform shuffle/draw mechanism.

The probability of at least one ace is simpler through the complement:
$$1-\frac{\binom{48}{5}}{\binom{52}{5}}\approx0.34116.$$
The same logic applies to samples from a finite set containing a designated type; the hypergeometric distribution in Module 4 formalizes it.

In [5]:
total_hands = math.comb(52, 5)
exactly_two_aces = Fraction(math.comb(4, 2)*math.comb(48, 3), total_hands)
at_least_one_ace = 1-Fraction(math.comb(48, 5), total_hands)
print("Exactly two aces:", float(exactly_two_aces))
print("At least one ace:", float(at_least_one_ace))
# Cross-check: probabilities over every possible ace count sum to 1.
ace_count_probabilities = [Fraction(math.comb(4, k)*math.comb(48, 5-k), total_hands)
                          for k in range(5)]
assert sum(ace_count_probabilities) == 1

Exactly two aces: 0.03992981808107859
At least one ace: 0.34115800166220334


### A common trap: unordered outcomes need not be equiprobable

Draw two fair binary symbols independently. Ordered outcomes AA,AB,BA,BB each have probability 1/4. Unordered multisets {A,A}, {A,B}, {B,B} have probabilities 1/4,1/2,1/4. There are three possible multisets, but the mixed outcome has two generating sequences.

Counting outcomes alone does not determine probabilities without a generating mechanism. A uniformly selected multiset would be a different experiment from two independent symbol draws.

In [6]:
from collections import Counter
ordered_pairs = list(product("AB", repeat=2))
multiset_frequencies = Counter(tuple(sorted(pair)) for pair in ordered_pairs)
print("Probabilities after forgetting order:",
      {key: Fraction(count, len(ordered_pairs)) for key, count in sorted(multiset_frequencies.items())})
assert multiset_frequencies[("A", "B")] == 2

Probabilities after forgetting order: {('A', 'A'): Fraction(1, 4), ('A', 'B'): Fraction(1, 2), ('B', 'B'): Fraction(1, 4)}


## 7. AI/ML applications and combinatorial growth

Choosing $r$ features from $n$ gives $\binom nr$ subsets. Every subset of $n$ features, including the empty set, gives $2^n$ possibilities because each feature is either included or excluded. This rapid growth explains why exhaustive feature searches can be impractical.

A grid with independent allowed choices of 3 model families, 4 regularization values, and 5 thresholds has 60 candidate configurations, but selecting the best on the test set would bias the reported evaluation. Search on training/validation data and reserve a final test evaluation.

For an idealized Naive Bayes model with $d$ binary features, there are $2^d$ feature patterns. Conditional independence reduces the need to estimate a separate joint probability for every pattern within each class, replacing that table with $d$ Bernoulli probabilities per class. This reduces complexity but imposes a modeling assumption; it does not eliminate the need for representative data or calibration checks.

In [7]:
print("Choose 3 features out of 20:", math.comb(20, 3))
print("All subsets of 20 features:", 2**20)
print("All binary patterns for 30 features:", 2**30)
print("Bernoulli feature parameters per class under Naive Bayes:", 30)
# Enumerate only a small case to verify the all-subsets identity.
assert sum(math.comb(5, r) for r in range(6)) == 2**5

Choose 3 features out of 20: 1140
All subsets of 20 features: 1048576
All binary patterns for 30 features: 1073741824
Bernoulli feature parameters per class under Naive Bayes: 30


## Practice

1. A menu offers 3 starters, 4 mains, and 2 drinks, all combinations allowed. How many meals with one of each?
2. How many ways can 3 ranked prizes be assigned among 8 people without repeats? How many 3-person committees can be chosen?
3. How many length-5 binary strings exist? How many contain exactly two ones?
4. How many distinct arrangements does “BANANA” have?
5. Select 4 items uniformly without replacement from 6 good and 4 defective items. Find the probability of exactly one defective item.
6. How many multisets of size 4 can be chosen from 3 types? Are they equally likely if four types are drawn independently and uniformly?
7. How many nonempty feature subsets exist for 10 distinct features?
8. Why must a probability numerator and denominator use the same outcome resolution?

## Solutions

1. $3\times4\times2=24$.
2. Ranked assignments $P(8,3)=336$; committees $\binom83=56$.
3. $2^5=32$ strings; choose the two one-positions in $\binom52=10$ ways.
4. A occurs 3 times, N twice, B once: $6!/(3!2!1!)=60$.
5. $\binom41\binom63/\binom{10}{4}=80/210=8/21\approx0.38095$.
6. $\binom{3+4-1}{4}=\binom64=15$. No: multiset probabilities depend on how many ordered draws generate each count allocation.
7. $2^{10}-1=1023$, subtracting the empty subset.
8. Each elementary outcome must receive consistent counting weight. For example, a favorable unordered pair over all ordered pairs omits its multiple orderings and underestimates the probability.

## Module 3 recap

Define the experiment and space, build events, apply union/intersection and complement rules, condition on the right population, and use counting only when its assumptions match the probability model. Bayes' theorem updates prior uncertainty using likelihood and evidence; Naive Bayes adds a conditional-independence approximation for tractable classification.

**Readiness check:** solve a count both by a formula and by tiny-case enumeration; explain why a finite list of possible outcomes need not be uniform; and identify when a probability model's assumptions could fail in ML deployment.

**Next:** Module 4 — Probability Distributions in the [main syllabus](../README.md).